In [1]:
import json
import pandas as pd
from pathlib import Path

in_path = Path("/home/dtesta/Mechanistic_DisAttention_VLMs/data/MAIA_MC_task_priorCheck_itempool1.json")
out_path = in_path.with_suffix(".csv")  # stesso nome, estensione .csv

def load_items(path: Path):
    """
    Carica sia:
    - JSONL (una dict per riga)
    - JSON "classico" (lista di dict: [ {...}, {...} ])
    """
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    # Caso JSON array
    if text[0] == "[":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("Il file sembra JSON ma non contiene una lista.")
        return data

    # Caso JSONL
    items = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            items.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"Errore JSON alla riga {line_no}: {e}") from e
    return items

items = load_items(in_path)

rows = []
for i, it in enumerate(items):
    video_id = it.get("video_id", "")
    link = it.get("link", "")

    # "40esimo elemento" = i == 39 (indice 0-based). Quindi:
    # - i < 39  -> primi 39 (1..39)
    # - i >= 39 -> dal 40esimo in poi (40..)
    if i < 40:
        caption = it.get("caption", "")
        foil = it.get("foil", "")
    else:
        caption = ""
        foil = ""

    rows.append({
        "video_id": video_id,
        "link": link,
        "caption": caption,
        "foil": foil,
    })

df = pd.DataFrame(rows, columns=["video_id", "link", "caption", "foil"])
df.to_csv(out_path, index=False, encoding="utf-8")

print(f"Salvato: {out_path}  |  righe: {len(df)}")
df.head(5)

Salvato: /home/dtesta/Mechanistic_DisAttention_VLMs/data/MAIA_MC_task_priorCheck_itempool1.csv  |  righe: 198


,video_id,link,caption,foil
0,video1,https://youtu.be/JG7p5YfCJqU,La scena si svolge di fianco ad una fontana,La scena si svolge di fianco ad un lago
1,video1,https://youtu.be/JG7p5YfCJqU,Sul bordo posteriore della fontana si erge un ...,Sul bordo posteriore della fontana si erge un ...
2,video2,https://youtu.be/MTN7eg55rJY,La fasciatura sulla zampa del cane è gialla.,La fasciatura sulla zampa del cane è bianca.
3,video2,https://youtu.be/MTN7eg55rJY,La persona sbriciola a terra una fetta biscott...,La persona sbriciola a terra un biscotto.
4,video3,https://youtu.be/vN8VPMLqUfY,La borsa attaccata sotto ad uno degli ombrello...,La borsa attaccatasotto ad uno degli ombrellon...
